# 1. Prepare imagery: inspect, fix the CRS, tile

Both models read the imagery in pieces: SAM 3 through tiles, Esri through its own chips inside larger chunks. This notebook covers what happens before either model runs: checking the raster, fixing a mislabelled coordinate system without resampling, and cutting overlapping tiles.

It runs on the synthetic demo data from `nbf demo`, so it needs no licensed imagery. The table gives the real Lewisville values next to the demo ones.

| Step | Demo | Lewisville (Nearmap, May 2026) |
|---|---|---|
| Imagery | 1,536 × 512 px, 0.25 m, EPSG:32614 | 166,037 × 127,183 px, 10.16 cm, EPSG:2276 (US survey feet) through a VRT |
| SAM 3 tiles | 256 px, 32 px overlap | 1,536 px, 192 px overlap: 6,538 tiles |
| Esri chunks | 512 px core, 64 px overlap | 10,240 px core, 512 px overlap: 150 chunks |

In [1]:
import json
import shutil
from pathlib import Path

from nearmap_buildings.cli import main as _nbf

# Run from the repository root or from notebooks/. Generated files go to notebook_runs/, which git ignores.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORK = REPO / "notebook_runs" / "01_prepare_imagery"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)


def nbf(*args):
    """Run one nbf command in this process; stop the notebook if it fails."""
    code = _nbf([str(a) for a in args])
    if code:
        raise RuntimeError(f"nbf {args[0]} failed with exit code {code}")

## Synthetic demo data

`nbf demo` writes a small synthetic image with nine flat roofs, their reference outlines, an area of interest, train/validation/test regions and two prediction files used later for scoring. None of it is Nearmap imagery or model output.

In [2]:
DEMO = WORK / "demo"
nbf("demo", "--output", DEMO)
sorted(p.name for p in DEMO.iterdir())

<repo>\notebook_runs\01_prepare_imagery\demo


['NOTICE.json',
 'aoi.geojson',
 'boxes.geojson',
 'esri_format_predictions.gdb',
 'exemplars.geojson',
 'imagery.tif',
 'perfect_predictions.gpkg',
 'points.geojson',
 'reference.gpkg',
 'splits.geojson']

## Inspect the raster

Check three things before anything else: the CRS is a real projected CRS, the pixel size is what the vendor states, and the centre falls where the imagery should be.

In [3]:
nbf("inspect", DEMO / "imagery.tif")

{
  "path": "<repo>\\notebook_runs\\01_prepare_imagery\\demo\\imagery.tif",
  "width": 1536,
  "height": 512,
  "crs": "EPSG:32614",
  "bounds": [
    500000.0,
    3599872.0,
    500384.0,
    3600000.0
  ],
  "resolution_in_crs_units": [
    0.25,
    0.25
  ],
  "bands": [
    1,
    2,
    3
  ],
  "dtypes": [
    "uint8",
    "uint8",
    "uint8"
  ],
  "nodata": null
}


In [4]:
import rasterio
from pyproj import Transformer


def centre(path):
    """Centre of a raster in latitude and longitude, with its pixel size and CRS."""
    with rasterio.open(path) as src:
        x = (src.bounds.left + src.bounds.right) / 2
        y = (src.bounds.bottom + src.bounds.top) / 2
        lon, lat = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True).transform(x, y)
        return {"lat": round(lat, 5), "lon": round(lon, 5), "pixel": src.res, "crs": src.crs.to_string()}


centre(DEMO / "imagery.tif")

{'lat': 32.53678, 'lon': -98.99796, 'pixel': (0.25, 0.25), 'crs': 'EPSG:32614'}

## Fix a mislabelled CRS without resampling

Lewisville's GeoTIFF declared a local coordinate system in metres, with no projection. Its coordinates were really EPSG:2276 (NAD83 / Texas North Central, US survey feet): read that way, the centre falls on Lewisville, the extent matches the city limits and a pixel is 1/3 ft, or 10.16 cm. Rewriting a 25 GB file was unnecessary. A small VRT points at the original pixels and declares the right CRS and nodata value; nothing is resampled.

Here the same helper relabels a copy of the demo image that has lost its CRS.

In [5]:
from xml.sax.saxutils import escape
from rasterio.crs import CRS


def relabel_vrt(source, vrt_path, crs, nodata=0):
    """Write a VRT that reads `source` pixel for pixel but declares `crs` and `nodata`."""
    with rasterio.open(source) as src:
        assert set(src.dtypes) == {"uint8"}, "this helper writes Byte bands"
        t = src.transform
        bands = "".join(
            f'<VRTRasterBand dataType="Byte" band="{b}"><NoDataValue>{nodata}</NoDataValue>'
            f'<SimpleSource><SourceFilename relativeToVRT="0">{escape(str(Path(source).resolve()))}</SourceFilename>'
            f"<SourceBand>{b}</SourceBand></SimpleSource></VRTRasterBand>"
            for b in range(1, src.count + 1))
        geo = f"{t.c!r}, {t.a!r}, {t.b!r}, {t.f!r}, {t.d!r}, {t.e!r}"
        Path(vrt_path).write_text(
            f'<VRTDataset rasterXSize="{src.width}" rasterYSize="{src.height}">'
            f"<SRS>{escape(CRS.from_user_input(crs).to_wkt())}</SRS><GeoTransform>{geo}</GeoTransform>{bands}</VRTDataset>")
    return Path(vrt_path)


# A copy of the demo image with its CRS stripped, as a stand-in for a mislabelled file.
with rasterio.open(DEMO / "imagery.tif") as src:
    profile, pixels = src.profile | {"crs": None}, src.read()
with rasterio.open(WORK / "no_crs.tif", "w", **profile) as dst:
    dst.write(pixels)

vrt = relabel_vrt(WORK / "no_crs.tif", WORK / "relabelled.vrt", "EPSG:32614")
with rasterio.open(vrt) as fixed:
    assert (fixed.read() == pixels).all(), "the VRT must return the original pixels"
centre(vrt)

{'lat': 32.53678, 'lon': -98.99796, 'pixel': (0.25, 0.25), 'crs': 'EPSG:32614'}

## Tile

`nbf tile` cuts overlapping windows at full resolution. It records each tile's bounds and SHA-256 in a manifest, and inference later refuses a tile that has changed. Tiles with less than 1% valid pixels are skipped, which drops the black fill outside Lewisville's city limits: 11,921 empty windows at 1,024 px.

Overlap matters twice: a building cut at one tile's edge is whole in its neighbour, and cleanup keeps that complete copy.

In [6]:
nbf("tile", vrt, "--output", WORK / "tiles_256", "--tile-size", 256, "--overlap", 32)

manifest = json.loads((WORK / "tiles_256" / "manifest.json").read_text())
print(len(manifest["tiles"]), "tiles of", manifest["tile_size"], "px with", manifest["overlap"], "px overlap")
manifest["tiles"][0]

Prepared 21 tiles: <repo>\notebook_runs\01_prepare_imagery\tiles_256\manifest.json
21 tiles of 256 px with 32 px overlap


{'id': 'r00000000_c00000000',
 'path': 'tiles/r00000000_c00000000.tif',
 'row_off': 0,
 'col_off': 0,
 'width': 256,
 'height': 256,
 'crs': 'EPSG:32614',
 'bounds': [500000.0, 3599936.0, 500064.0, 3600000.0],
 'transform': [0.25, 0.0, 500000.0, 0.0, -0.25, 3600000.0],
 'valid_fraction': 1.0,
 'sha256': 'd3bb2198041bcb90a443e3d1530d19aac41c9f5fb40528f653a3d58cfce41338'}

For Lewisville the same two steps ran on the relabelled VRT:

```bash
nbf tile lewisville_epsg2276.vrt --output tiles_1536 --tile-size 1536 --overlap 192
```

That wrote 6,538 tiles in 20 minutes. Next: [2. Run SAM 3](02_run_sam3.ipynb).